**MPNST RNA-seq Data Discovery and Manifest**

This notebook is the planning/discovery step for the MPNST TE-counts analysis.
Aim is to create a clean sample manifest before downloading or processing large RNA-seq files -> should prevent accidental over-downloading and make the later TE-counting notebooks reproducible.

Paper being followed up:
Xiao et al. 2025, Cancers, "A Sequencing Overview of Malignant Peripheral Nerve Sheath Tumors"
- That review lists several MPNST sequencing datasets, including bulk RNA-seq cohorts. 

Key Question: Do PRC2-deficient MPNST samples show ERV/TE derepression compared with PRC2-intact MPNST samples?
- Important: this notebook does not download raw data automatically. First use to record which dataset, accession, file type and metadata are available.

In [ ]:
# Load libraries and set paths

library(dplyr)
library(readr)
library(tibble)
library(stringr)
library(purrr)

# All four MPNST notebooks use this shared analysis directory
project_root <- "/home/kennes38/ResearchProject"
mpnst_root <- file.path(project_root, "05_MPNST_TE_analysis")
dir.create(mpnst_root, showWarnings = FALSE, recursive = TRUE)
setwd(mpnst_root)

# Keep small metadata/manifests here
manifest_dir <- file.path(mpnst_root, "manifests")
dir.create(manifest_dir, showWarnings = FALSE, recursive = TRUE)

raw_data_dir <- file.path(mpnst_root, "raw_or_external_data")
dir.create(raw_data_dir, showWarnings = FALSE, recursive = TRUE)

# TE count outputs from later notebooks
counts_dir <- file.path(mpnst_root, "te_counts")
dir.create(counts_dir, showWarnings = FALSE, recursive = TRUE)

getwd()

In [ ]:
# Record candidate MPNST dataset sources


candidate_datasets <- tibble::tribble(
  ~dataset_label, ~source_reference, ~accession_or_link, ~sample_context, ~data_type, ~access_notes,
  "GeM_Consortium_MPNST", "Xiao et al. 2025 review; GeM Consortium 2020", "EGAD00001008608", "Human MPNST tissue", "Bulk RNA-seq", "Likely controlled access; confirm availability and metadata",
  "Chi_2022_MPNST", "Xiao et al. 2025 review; Chi et al. 2022", "GSE206527 / GSE179699", "Human MPNST tissue", "Bulk RNA-seq", "Check GEO/SRA for FASTQ or processed counts",
  "Wu_2022_MPNST", "Xiao et al. 2025 review; Wu et al. 2022", "GSE178989 / GSE179033 / GSE179041", "Human tissue PN vs MPNST", "Bulk RNA-seq / scRNA-seq / ATAC-seq", "May include useful comparison samples; check sample metadata"
)

write_csv(candidate_datasets, file.path(manifest_dir, "candidate_MPNST_RNAseq_datasets.csv"))

candidate_datasets

In [ ]:
# Create an empty sample table manifest

# Key file for MPNST analysis - one row per RNA-seq sample/file.

sample_manifest_template <- tibble(
  sample_id = character(),
  patient_id = character(),
  dataset_label = character(),
  accession = character(),
  file_type = character(),
  fastq_1 = character(),
  fastq_2 = character(),
  bam_file = character(),
  processed_counts_file = character(),
  tumour_type = character(),
  sample_type = character(),
  PRC2_status = character(),
  PRC2_basis = character(),
  EED_status = character(),
  SUZ12_status = character(),
  H3K27me3_status = character(),
  NF1_status = character(),
  notes = character()
)

manifest_file <- file.path(manifest_dir, "MPNST_RNAseq_sample_manifest_TEMPLATE.csv")
write_csv(sample_manifest_template, manifest_file)

manifest_file

In [ ]:
# Load SRA Metadata and Build the Analysis Manifest

# Save the downloaded SRA Run Selector table under this filename
metadata_file <- file.path(manifest_dir, "GSE206527_SraRunTable.csv")

if (!file.exists(metadata_file)) {
  stop(
    "SRA metadata file not found. Move SraRunTable.csv to: ",
    metadata_file
  )
}

metadata <- read_csv(metadata_file, show_col_types = FALSE)

# Check that the fields needed to build the manifest are present
required_metadata_columns <- c(
  "Run", "BioSample", "Experiment", "genotype", "Library Name",
  "LibraryLayout", "Assay Type", "Organism", "source_name", "tissue", "Bytes"
)

missing_metadata_columns <- setdiff(required_metadata_columns, names(metadata))
if (length(missing_metadata_columns) > 0) {
  stop(
    "SRA metadata is missing required columns: ",
    paste(missing_metadata_columns, collapse = ", ")
  )
}

cat("Metadata rows:", nrow(metadata), "\n")
print(metadata %>% count(genotype, LibraryLayout))

# Validate the expected GSE206527 study composition before creating files
stopifnot(nrow(metadata) == 41)
stopifnot(n_distinct(metadata$Run) == 41)
stopifnot(all(metadata$LibraryLayout == "PAIRED"))
stopifnot(all(metadata$`Assay Type` == "RNA-Seq"))
stopifnot(all(metadata$Organism == "Homo sapiens"))
stopifnot(sum(metadata$genotype == "PRC2 loss of function") == 26)
stopifnot(sum(metadata$genotype == "PRC2 intact") == 15)

# Construct the manifest used by the TE-counting notebooks
sample_manifest <- metadata %>%
  transmute(
    sample_id = Run,
    patient_id = `Sample Name`,
    dataset_label = "GSE206527_Chi_MPNST",
    accession = Run,
    geo_accession = `Sample Name`,
    biosample_accession = BioSample,
    experiment_accession = Experiment,
    file_type = "FASTQ_PE",

    # These are expected local paths after paired FASTQ download.
    fastq_1 = file.path(raw_data_dir, Run, paste0(Run, "_1.fastq.gz")),
    fastq_2 = file.path(raw_data_dir, Run, paste0(Run, "_2.fastq.gz")),
    bam_file = NA_character_,
    processed_counts_file = NA_character_,
    tumour_type = "MPNST",
    sample_type = "primary_tumour",
    PRC2_status = case_when(
      genotype == "PRC2 loss of function" ~ "PRC2_loss",
      genotype == "PRC2 intact" ~ "PRC2_intact",
      TRUE ~ "unknown"
    ),
    PRC2_basis = genotype,
    EED_status = NA_character_,
    SUZ12_status = NA_character_,
    H3K27me3_status = NA_character_,
    NF1_status = NA_character_,
    library_layout = LibraryLayout,
    estimated_download_gb = Bytes / 1e9,
    notes = paste(`Library Name`, source_name, tissue, sep = "; ")
  ) %>%
  arrange(PRC2_status, sample_id)

final_manifest_file <- file.path(manifest_dir, "MPNST_RNAseq_sample_manifest.csv")
write_csv(sample_manifest, final_manifest_file)

# Build a storage inventory directly from the SRA metadata
file_inventory <- sample_manifest %>%
  transmute(
    sample_id,
    file_path_or_url = paste0("SRA:", accession),
    file_type,
    size_gb = estimated_download_gb,
    notes = paste0("Paired-end run; GEO sample ", geo_accession)
  )

final_inventory_file <- file.path(manifest_dir, "MPNST_RNAseq_file_inventory.csv")
write_csv(file_inventory, final_inventory_file)

cat("Manifest written to:", final_manifest_file, "\n")
cat("Samples in manifest:", nrow(sample_manifest), "\n")
print(sample_manifest %>% count(PRC2_status))
cat(
  "Approximate compressed SRA download size:",
  round(sum(sample_manifest$estimated_download_gb), 1),
  "GB\n"
)

sample_manifest %>%
  select(sample_id, geo_accession, PRC2_status, file_type, estimated_download_gb) %>%
  slice_head(n = 10)

In [ ]:
# Estimate full file size before downloading

file_inventory_template <- tibble(
  sample_id = character(),
  file_path_or_url = character(),
  file_type = character(),
  size_gb = numeric(),
  notes = character()
)

inventory_file <- file.path(manifest_dir, "MPNST_RNAseq_file_inventory_TEMPLATE.csv")
write_csv(file_inventory_template, inventory_file)

# Once populated, block will summarise expected storage
if (file.exists(file.path(manifest_dir, "MPNST_RNAseq_file_inventory.csv"))) {
  file_inventory <- read_csv(file.path(manifest_dir, "MPNST_RNAseq_file_inventory.csv"), show_col_types = FALSE)
  file_inventory %>%
    summarise(
      n_files = n(),
      n_samples = n_distinct(sample_id),
      total_size_gb = sum(size_gb, na.rm = TRUE),
      estimated_working_space_gb = total_size_gb * 2
    )
} else {
  message("File inventory template written to: ", inventory_file)
}